# Covariance Shrinkage in Cross-Asset ETF Portfolios

**Research question.** Does Ledoit-Wolf covariance shrinkage improve out-of-sample risk and weight stability relative to sample-covariance minimum variance?

I evaluate **3,270** daily observations from **2013-01-02** to **2025-12-31**. I keep the core logic in `src/` and use this notebook only to load and present reproducible artifacts.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
TABLES = PROJECT_ROOT / 'reports' / 'tables'
performance = pd.read_csv(
    TABLES / 'main_performance.csv', index_col='strategy'
)
sensitivity = pd.read_csv(TABLES / 'sensitivity.csv')
var_backtest = pd.read_csv(
    TABLES / 'var_backtest.csv', index_col='strategy'
)
findings = json.loads(
    (TABLES / 'research_findings.json').read_text(encoding='utf-8')
)
print(f"OOS: {findings['evaluation_start']} to {findings['evaluation_end']}")

OOS: 2013-01-02 to 2025-12-31


## Design

I evaluate nine cross-asset ETFs with quarterly walk-forward decisions. I use a 252-day training window, 40% asset cap, and 10 bp costs on traded notional in the main configuration. I restrict every target to information available through the prior trading day. I use buy-and-hold 1/N, quarterly 1/N, inverse volatility, and SPY as baselines.

In [2]:
columns = ['cagr', 'annual_volatility', 'sharpe', 'max_drawdown', 'annualized_turnover']
main = performance[columns].copy()
main[['cagr', 'annual_volatility', 'max_drawdown']] *= 100
main.columns = ['CAGR (%)', 'Volatility (%)', 'Sharpe', 'Max drawdown (%)', 'Annual turnover']
display(main.round(3))

,CAGR (%),Volatility (%),Sharpe,Max drawdown (%),Annual turnover
strategy,,,,,
buy_and_hold_equal_weight,6.125,10.145,0.637,-22.955,0.077
quarterly_equal_weight,5.443,9.145,0.625,-20.530,0.239
inverse_volatility,4.474,7.575,0.616,-20.562,0.327
sample_min_variance,3.503,6.232,0.584,-19.232,0.652
ledoit_wolf_min_variance,3.465,6.237,0.577,-19.409,0.628
spy,14.795,16.936,0.900,-33.717,0.077


## Finding

I find that Ledoit-Wolf reduced the median covariance condition number by **65.9%** and annual turnover by **0.024**. It did **not** improve main-setting risk: annual volatility was 6.24% versus 6.23%, and maximum drawdown was -19.41% versus -19.23%. I observe lower volatility in only **3 of 9** lookback-cost scenarios.

## Performance and drawdowns

![Net performance and drawdowns](../../reports/figures/performance_and_drawdowns.png)

## Conditioning and concentration

![Covariance diagnostics](../../reports/figures/covariance_diagnostics.png)

## Robustness

I use the same 2013-2025 test dates for every 126/252/504-day lookback and 0/10/25 bp cost setting. Positive cells mean shrinkage had higher risk.

![Sensitivity](../../reports/figures/sensitivity_heatmaps.png)

In [3]:
min_var = sensitivity[sensitivity['strategy'].isin([
    'sample_min_variance', 'ledoit_wolf_min_variance'
])][['lookback', 'cost_bps', 'strategy', 'annual_volatility', 'max_drawdown', 'annualized_turnover']]
display(min_var.round(5))

,lookback,cost_bps,strategy,annual_volatility,max_drawdown,annualized_turnover
3,126,0.0,sample_min_variance,0.06248,-0.19699,0.98139
4,126,0.0,ledoit_wolf_min_variance,0.06261,-0.19904,0.99898
9,126,10.0,sample_min_variance,0.06247,-0.19794,0.98139
10,126,10.0,ledoit_wolf_min_variance,0.06260,-0.19996,0.99898
15,126,25.0,sample_min_variance,0.06246,-0.19937,0.98139
16,126,25.0,ledoit_wolf_min_variance,0.06259,-0.20133,0.99898
21,252,0.0,sample_min_variance,0.06234,-0.19181,0.65240
22,252,0.0,ledoit_wolf_min_variance,0.06238,-0.19357,0.62829
27,252,10.0,sample_min_variance,0.06232,-0.19232,0.65240
28,252,10.0,ledoit_wolf_min_variance,0.06237,-0.19409,0.62829


## VaR validation

I calculate historical 95% VaR forecasts from the prior 252 strategy returns and exclude the current day. I use Kupiec tests to assess unconditional breach frequency.

In [4]:
display(var_backtest[['forecast_observations', 'breaches', 'breach_rate', 'kupiec_p_value']].round(4))

,forecast_observations,breaches,breach_rate,kupiec_p_value
strategy,,,,
buy_and_hold_equal_weight,3018,169,0.0560,0.1377
quarterly_equal_weight,3018,173,0.0573,0.0710
inverse_volatility,3018,171,0.0567,0.0999
sample_min_variance,3018,161,0.0533,0.4038
ledoit_wolf_min_variance,3018,166,0.0550,0.2142
spy,3018,161,0.0533,0.4038


## Limitations

I select the ETF universe with hindsight. I use adjusted daily closes and omit intraday execution, spreads, market impact, taxes, and fund closure risk. My nine liquid ETFs are also a low-dimensional setting in which covariance shrinkage may have limited scope to help. I interpret the experiment as evidence of better conditioning, not a universal performance claim. I do not estimate statistical uncertainty around the small difference between the two minimum-variance portfolios.